In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Load the real reservoir dataset
df = pd.read_csv("data/reservoir_daily.csv")

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Select Mettur Reservoir
mettur = df[df["reservoir_slug"] == "mettur"].copy()
mettur = mettur.sort_values("date").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Mettur records:", len(mettur))
print("Date range:", mettur["date"].min().date(), "to", mettur["date"].max().date())

display(mettur.head())

Dataset loaded successfully!
Mettur records: 570
Date range: 2025-01-01 to 2026-10-25


,date,reservoir_slug,reservoir,group,full_depth_ft,capacity_mcft,level_ft,storage_mcft,inflow_cusecs,outflow_cusecs,last_year_level_ft,last_year_storage_mcft,source_updated
0,2025-01-01,mettur,Mettur,Tamil Nadu,120.0,93470,120.00,93470.0,1791.0,969.0,71.27,33801.0,2025-01-01
1,2025-01-02,mettur,Mettur,Tamil Nadu,120.0,93470,120.00,93470.0,1871.0,1605.0,71.27,33801.0,2025-01-02
2,2025-01-03,mettur,Mettur,Tamil Nadu,120.0,93470,119.76,93089.0,1992.0,6136.0,71.27,33801.0,2025-01-03
3,2025-01-04,mettur,Mettur,Tamil Nadu,120.0,93470,119.14,92105.0,1128.0,12251.0,71.25,33783.0,2025-01-04
4,2025-01-05,mettur,Mettur,Tamil Nadu,120.0,93470,118.52,91130.0,1307.0,12326.0,71.23,33766.0,2025-01-05


In [2]:

# Check missing values in Mettur data
print("MISSING VALUES")
display(mettur.isnull().sum().to_frame("Missing Count"))

# Check duplicate records
print("DUPLICATE ROWS:", mettur.duplicated().sum())

# Check for impossible or suspicious values
print("\nSTORAGE AND CAPACITY CHECK")
print("Minimum storage:", mettur["storage_mcft"].min(), "MCFT")
print("Maximum storage:", mettur["storage_mcft"].max(), "MCFT")
print("Reservoir capacity:", mettur["capacity_mcft"].max(), "MCFT")

print("\nWATER LEVEL RANGE")
print("Minimum level:", mettur["level_ft"].min(), "ft")
print("Maximum level:", mettur["level_ft"].max(), "ft")

# Check whether dates are missing
all_dates = pd.date_range(
    start=mettur["date"].min(),
    end=mettur["date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(mettur["date"])

print("\nEXPECTED DAILY RECORDS:", len(all_dates))
print("ACTUAL RECORDS:", len(mettur))
print("MISSING DATES:", len(missing_dates))


MISSING VALUES


,Missing Count
date,0
reservoir_slug,0
reservoir,0
group,0
full_depth_ft,0
capacity_mcft,0
level_ft,0
storage_mcft,0
inflow_cusecs,0
outflow_cusecs,0


DUPLICATE ROWS: 0

STORAGE AND CAPACITY CHECK
Minimum storage: 20805.0 MCFT
Maximum storage: 93678.0 MCFT
Reservoir capacity: 93470 MCFT

WATER LEVEL RANGE
Minimum level: 54.56 ft
Maximum level: 120.13 ft

EXPECTED DAILY RECORDS: 663
ACTUAL RECORDS: 570
MISSING DATES: 93


In [6]:
%pip install nbformat ipython

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import nbformat
import IPython

print("nbformat:", nbformat.__version__)
print("IPython:", IPython.__version__)

nbformat: 5.11.1
IPython: 9.17.1


In [12]:

import plotly.express as px

fig = px.line(
    mettur,
    x="date",
    y="storage_mcft",
    title="Mettur Reservoir Storage Over Time",
    labels={
        "date": "Date",
        "storage_mcft": "Storage (MCFT)"
    },
    hover_data=["level_ft", "inflow_cusecs", "outflow_cusecs"]
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

from IPython.display import display, HTML

display(HTML(fig.to_html(include_plotlyjs="cdn")))


In [13]:

# 1. Calculate storage as a percentage of capacity
mettur["storage_percentage"] = (
    mettur["storage_mcft"] / mettur["capacity_mcft"]
) * 100

# 2. Summarize storage conditions
print("METTUR STORAGE ANALYSIS")
print("-" * 35)
print(f"Average storage: {mettur['storage_mcft'].mean():,.2f} MCFT")
print(f"Average storage level: {mettur['storage_percentage'].mean():.2f}%")
print(f"Lowest storage: {mettur['storage_mcft'].min():,.2f} MCFT")
print(f"Highest storage: {mettur['storage_mcft'].max():,.2f} MCFT")

# 3. Find the five lowest-storage days
print("\nFIVE LOWEST-STORAGE DAYS")
display(
    mettur.nsmallest(5, "storage_mcft")[
        ["date", "storage_mcft", "storage_percentage", "level_ft"]
    ].reset_index(drop=True)
)

# 4. Find the five highest-storage days
print("\nFIVE HIGHEST-STORAGE DAYS")
display(
    mettur.nlargest(5, "storage_mcft")[
        ["date", "storage_mcft", "storage_percentage", "level_ft"]
    ].reset_index(drop=True)
)


METTUR STORAGE ANALYSIS
-----------------------------------
Average storage: 66,027.63 MCFT
Average storage level: 70.64%
Lowest storage: 20,805.00 MCFT
Highest storage: 93,678.00 MCFT

FIVE LOWEST-STORAGE DAYS


,date,storage_mcft,storage_percentage,level_ft
0,2026-10-24,20805.0,22.258479,54.56
1,2026-10-25,20846.0,22.302343,54.62
2,2026-10-23,20887.0,22.346207,54.68
3,2026-10-22,20955.0,22.418958,54.78
4,2026-10-21,21044.0,22.514176,54.91



FIVE HIGHEST-STORAGE DAYS


,date,storage_mcft,storage_percentage,level_ft
0,2025-08-20,93678.0,100.222531,120.13
1,2025-01-01,93470.0,100.000000,120.00
2,2025-01-02,93470.0,100.000000,120.00
3,2025-06-30,93470.0,100.000000,120.00
4,2025-07-01,93470.0,100.000000,120.00


In [ ]:

# Identify dates missing from the Mettur dataset
all_dates = pd.date_range(
    start=mettur["date"].min(),
    end=mettur["date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(mettur["date"])

print("Total missing dates:", len(missing_dates))
print("\nFirst 20 missing dates:")
display(pd.DataFrame({"missing_date": missing_dates[:20]}))

# Find storage values above the recorded capacity
over_capacity = mettur[
    mettur["storage_mcft"] > mettur["capacity_mcft"]
][
    ["date", "storage_mcft", "capacity_mcft", "level_ft"]
]

print("\nRecords exceeding stated capacity:")
display(over_capacity)

# Check whether dates are consecutive in the original data
mettur["days_between_records"] = mettur["date"].diff().dt.days

print("\nLargest gaps between consecutive records:")
display(
    mettur.loc[
        mettur["days_between_records"] > 1,
        ["date", "days_between_records"]
    ].head(15)
)

Total missing dates: 93

First 20 missing dates:


,missing_date
0,2025-05-08
1,2025-05-09
2,2025-05-10
3,2025-05-11
4,2025-05-12
5,2025-05-13
6,2025-07-27
7,2025-08-02
8,2025-08-03
9,2025-08-04



Records exceeding stated capacity:


,date,storage_mcft,capacity_mcft,level_ft
220,2025-08-20,93678.0,93470,120.13



Largest gaps between consecutive records:


,date,days_between_records
127,2025-05-14,7.0
201,2025-07-28,2.0
206,2025-08-05,4.0
210,2025-08-10,2.0
224,2025-09-01,9.0
227,2025-09-08,5.0
239,2025-09-21,2.0
249,2025-10-02,2.0
258,2025-10-13,3.0
260,2025-10-24,10.0


In [ ]:

# Work on a separate copy; preserve the original data
water_balance = mettur.copy()

# Convert daily average flow (cusecs) to daily volume (MCFT)
SECONDS_PER_DAY = 24 * 60 * 60
CUBIC_FEET_PER_MILLION = 1_000_000

water_balance["inflow_mcft_day"] = (
    water_balance["inflow_cusecs"]
    * SECONDS_PER_DAY
    / CUBIC_FEET_PER_MILLION
)

water_balance["outflow_mcft_day"] = (
    water_balance["outflow_cusecs"]
    * SECONDS_PER_DAY
    / CUBIC_FEET_PER_MILLION
)

# Change in observed storage between consecutive records
water_balance["storage_change_mcft"] = (
    water_balance["storage_mcft"].diff()
)

# Net flow estimated from reported inflow and outflow
water_balance["net_flow_mcft_day"] = (
    water_balance["inflow_mcft_day"]
    - water_balance["outflow_mcft_day"]
)

# Only compare records exactly one day apart
water_balance["days_gap"] = water_balance["date"].diff().dt.days

daily_records = water_balance[
    water_balance["days_gap"] == 1
].copy()

print("Consecutive-day comparisons:", len(daily_records))
print(
    "Average observed storage change:",
    round(daily_records["storage_change_mcft"].mean(), 2),
    "MCFT/day"
)
print(
    "Average reported net flow:",
    round(daily_records["net_flow_mcft_day"].mean(), 2),
    "MCFT/day"
)

print("\nSample water-balance records:")
display(
    daily_records[
        [
            "date",
            "storage_change_mcft",
            "inflow_mcft_day",
            "outflow_mcft_day",
            "net_flow_mcft_day"
        ]
    ].head(10)
)

Consecutive-day comparisons: 525
Average observed storage change: -110.86 MCFT/day
Average reported net flow: -85.35 MCFT/day

Sample water-balance records:


,date,storage_change_mcft,inflow_mcft_day,outflow_mcft_day,net_flow_mcft_day
1,2025-01-02,0.0,161.6544,138.6720,22.9824
2,2025-01-03,-381.0,172.1088,530.1504,-358.0416
3,2025-01-04,-984.0,97.4592,1058.4864,-961.0272
4,2025-01-05,-975.0,112.9248,1064.9664,-952.0416
5,2025-01-06,-1017.0,71.5392,1065.5712,-994.0320
6,2025-01-07,-1023.0,64.6272,1065.5712,-1000.9440
7,2025-01-08,-862.0,59.9616,899.9424,-839.9808
8,2025-01-09,-843.0,71.7984,892.7712,-820.9728
9,2025-01-10,-683.0,65.4912,294.4512,-228.9600
10,2025-01-11,-515.0,64.3680,557.3664,-492.9984


In [16]:

import plotly.graph_objects as go
from IPython.display import display, HTML

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_records["date"],
    y=daily_records["storage_change_mcft"],
    mode="lines",
    name="Observed Storage Change",
    line=dict(width=1.5)
))

fig.add_trace(go.Scatter(
    x=daily_records["date"],
    y=daily_records["net_flow_mcft_day"],
    mode="lines",
    name="Reported Net Flow",
    line=dict(width=1.5)
))

fig.update_layout(
    title="Mettur Reservoir: Storage Change vs Reported Net Flow",
    xaxis_title="Date",
    yaxis_title="Water Volume (MCFT/day)",
    template="plotly_white",
    hovermode="x unified"
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))


In [17]:

# Calculate the correlation between the two measurements
correlation = daily_records[
    ["storage_change_mcft", "net_flow_mcft_day"]
].corr().iloc[0, 1]

print(f"Correlation: {correlation:.3f}")

# Plot the two measurements in separate panels for easier comparison
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_records["date"],
    y=daily_records["storage_change_mcft"],
    mode="lines",
    name="Observed Storage Change"
))

fig.add_trace(go.Scatter(
    x=daily_records["date"],
    y=daily_records["net_flow_mcft_day"],
    mode="lines",
    name="Reported Net Flow"
))

fig.update_layout(
    title="Mettur Reservoir Water Balance",
    template="plotly_white",
    height=600,
    width=1100,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.08, x=0),
    margin=dict(l=60, r=30, t=100, b=70),
    xaxis=dict(title="Date", rangeslider=dict(visible=True)),
    yaxis=dict(title="Water volume (MCFT/day)")
)

display(HTML(fig.to_html(
    include_plotlyjs="cdn",
    default_width="100%",
    default_height="600px"
)))


Correlation: 0.999


In [ ]:

# Compare observed storage changes with reported net flow
daily_records["balance_difference_mcft"] = (
    daily_records["storage_change_mcft"]
    - daily_records["net_flow_mcft_day"]
)

print("WATER BALANCE COMPARISON")
print("-" * 35)

print(
    "Mean difference:",
    round(daily_records["balance_difference_mcft"].mean(), 2),
    "MCFT/day"
)

print(
    "Mean absolute difference:",
    round(daily_records["balance_difference_mcft"].abs().mean(), 2),
    "MCFT/day"
)

print(
    "Maximum absolute difference:",
    round(daily_records["balance_difference_mcft"].abs().max(), 2),
    "MCFT/day"
)

print("\nLargest differences:")
display(
    daily_records.loc[
        daily_records["balance_difference_mcft"].abs().nlargest(5).index,
        [
            "date",
            "storage_change_mcft",
            "net_flow_mcft_day",
            "balance_difference_mcft"
        ]
    ].sort_values("balance_difference_mcft", key=abs, ascending=False)
)

WATER BALANCE COMPARISON
-----------------------------------
Mean difference: -25.5 MCFT/day
Mean absolute difference: 25.5 MCFT/day
Maximum absolute difference: 454.04 MCFT/day

Largest differences:


,date,storage_change_mcft,net_flow_mcft_day,balance_difference_mcft
9,2025-01-10,-683.0,-228.9600,-454.0400
247,2025-09-29,-753.0,-466.8192,-286.1808
491,2026-08-03,-140.0,-80.7840,-59.2160
60,2025-03-02,-87.0,-50.9760,-36.0240
98,2025-04-09,41.0,75.0816,-34.0816


In [19]:

# Create a cleaned analysis copy of Mettur data
mettur_clean = mettur.copy()

# Ensure dates are in datetime format
mettur_clean["date"] = pd.to_datetime(mettur_clean["date"])

# Calculate storage as a percentage of stated capacity
mettur_clean["storage_percentage"] = (
    mettur_clean["storage_mcft"]
    / mettur_clean["capacity_mcft"]
    * 100
)

# Flag records exceeding stated capacity
mettur_clean["over_capacity_flag"] = (
    mettur_clean["storage_mcft"]
    > mettur_clean["capacity_mcft"]
)

# Sort by date
mettur_clean = mettur_clean.sort_values("date").reset_index(drop=True)

print("Analysis dataset prepared.")
print("Records:", len(mettur_clean))
print("Missing dates remain unfilled:", len(missing_dates))
print(
    "Over-capacity records:",
    int(mettur_clean["over_capacity_flag"].sum())
)

display(
    mettur_clean[
        [
            "date",
            "storage_mcft",
            "capacity_mcft",
            "storage_percentage",
            "over_capacity_flag"
        ]
    ].head()
)


Analysis dataset prepared.
Records: 570
Missing dates remain unfilled: 93
Over-capacity records: 1


,date,storage_mcft,capacity_mcft,storage_percentage,over_capacity_flag
0,2025-01-01,93470.0,93470,100.000000,False
1,2025-01-02,93470.0,93470,100.000000,False
2,2025-01-03,93089.0,93470,99.592383,False
3,2025-01-04,92105.0,93470,98.539638,False
4,2025-01-05,91130.0,93470,97.496523,False


In [24]:

from src.allocation import allocate_water

# Illustrative planning scenario, not official sector demand data
result = allocate_water(
    available_water=100,
    domestic_demand=30,
    agriculture_demand=60,
    industry_demand=20,
    reserve_water=20
)

for key, value in result.items():
    print(f"{key}: {value:.2f}")


Available Water (MCFT): 100.00
Protected Reserve (MCFT): 20.00
Usable Water (MCFT): 80.00
Total Demand (MCFT): 110.00
Total Allocated (MCFT): 80.00
Unmet Demand (MCFT): 30.00
Domestic Demand (MCFT): 30.00
Domestic Allocation (MCFT): 30.00
Domestic Unmet (MCFT): 0.00
Agriculture Demand (MCFT): 60.00
Agriculture Allocation (MCFT): 50.00
Agriculture Unmet (MCFT): 10.00
Industry Demand (MCFT): 20.00
Industry Allocation (MCFT): 0.00
Industry Unmet (MCFT): 20.00


In [23]:
import importlib
import src.allocation as allocation_module

importlib.reload(allocation_module)

allocate_water = allocation_module.allocate_water

In [27]:
result = allocate_water(
    available_water=150,
    domestic_demand=30,
    agriculture_demand=60,
    industry_demand=20,
    reserve_water=20
)

for key, value in result.items():
    print(f"{key}: {value:.2f}")

Available Water (MCFT): 150.00
Protected Reserve (MCFT): 20.00
Usable Water (MCFT): 130.00
Total Demand (MCFT): 110.00
Total Allocated (MCFT): 110.00
Unmet Demand (MCFT): 0.00
Domestic Demand (MCFT): 30.00
Domestic Allocation (MCFT): 30.00
Domestic Unmet (MCFT): 0.00
Agriculture Demand (MCFT): 60.00
Agriculture Allocation (MCFT): 60.00
Agriculture Unmet (MCFT): 0.00
Industry Demand (MCFT): 20.00
Industry Allocation (MCFT): 20.00
Industry Unmet (MCFT): 0.00


In [30]:
import importlib
import src.allocation as allocation_module

importlib.reload(allocation_module)

result = allocation_module.optimize_water_allocation(
    available_water=100,
    domestic_demand=30,
    agriculture_demand=60,
    industry_demand=20,
    reserve_water=20
)

for key, value in result.items():
    print(f"{key}: {value:.2f}")

Available Water (MCFT): 100.00
Protected Reserve (MCFT): 20.00
Usable Water (MCFT): 80.00
Total Demand (MCFT): 110.00
Total Allocated (MCFT): 80.00
Unmet Demand (MCFT): 30.00
Domestic Demand (MCFT): 30.00
Domestic Allocation (MCFT): 30.00
Domestic Unmet (MCFT): 0.00
Agriculture Demand (MCFT): 60.00
Agriculture Allocation (MCFT): 50.00
Agriculture Unmet (MCFT): 10.00
Industry Demand (MCFT): 20.00
Industry Allocation (MCFT): 0.00
Industry Unmet (MCFT): 20.00


In [31]:
import importlib
import src.allocation as allocation_module

importlib.reload(allocation_module)

result = allocation_module.optimize_water_allocation(
    available_water=100,
    domestic_demand=30,
    agriculture_demand=60,
    industry_demand=20,
    reserve_water=20
)

for key, value in result.items():
    print(f"{key}: {value:.2f}")

Available Water (MCFT): 100.00
Protected Reserve (MCFT): 20.00
Usable Water (MCFT): 80.00
Total Demand (MCFT): 110.00
Total Allocated (MCFT): 80.00
Unmet Demand (MCFT): 30.00
Domestic Demand (MCFT): 30.00
Domestic Allocation (MCFT): 30.00
Domestic Unmet (MCFT): 0.00
Domestic Satisfaction (%): 100.00
Agriculture Demand (MCFT): 60.00
Agriculture Allocation (MCFT): 30.00
Agriculture Unmet (MCFT): 30.00
Agriculture Satisfaction (%): 50.00
Industry Demand (MCFT): 20.00
Industry Allocation (MCFT): 20.00
Industry Unmet (MCFT): 0.00
Industry Satisfaction (%): 100.00


In [32]:

from src.allocation import allocate_water, optimize_water_allocation
import pandas as pd

inputs = {
    "available_water": 100,
    "domestic_demand": 30,
    "agriculture_demand": 60,
    "industry_demand": 20,
    "reserve_water": 20,
}

baseline = allocate_water(**inputs)
optimized = optimize_water_allocation(**inputs)

comparison = pd.DataFrame({
    "Sector": ["Domestic", "Agriculture", "Industry"],
    "Demand (MCFT)": [
        baseline["Domestic Demand (MCFT)"],
        baseline["Agriculture Demand (MCFT)"],
        baseline["Industry Demand (MCFT)"],
    ],
    "Baseline Allocation (MCFT)": [
        baseline["Domestic Allocation (MCFT)"],
        baseline["Agriculture Allocation (MCFT)"],
        baseline["Industry Allocation (MCFT)"],
    ],
    "Optimized Allocation (MCFT)": [
        optimized["Domestic Allocation (MCFT)"],
        optimized["Agriculture Allocation (MCFT)"],
        optimized["Industry Allocation (MCFT)"],
    ],
    "Optimized Satisfaction (%)": [
        optimized["Domestic Satisfaction (%)"],
        optimized["Agriculture Satisfaction (%)"],
        optimized["Industry Satisfaction (%)"],
    ],
})

display(comparison.round(2))


,Sector,Demand (MCFT),Baseline Allocation (MCFT),Optimized Allocation (MCFT),Optimized Satisfaction (%)
0,Domestic,30.0,30.0,30.0,100.0
1,Agriculture,60.0,50.0,30.0,50.0
2,Industry,20.0,0.0,20.0,100.0


In [33]:

def calculate_metrics(result, priorities=None):
    if priorities is None:
        priorities = {
            "Domestic": 3.0,
            "Agriculture": 2.0,
            "Industry": 1.0,
        }

    sectors = ["Domestic", "Agriculture", "Industry"]
    total_demand = sum(
        result[f"{sector} Demand (MCFT)"]
        for sector in sectors
    )

    weighted_satisfaction = 0.0
    total_weight = 0.0

    for sector in sectors:
        demand = result[f"{sector} Demand (MCFT)"]
        allocation = result[f"{sector} Allocation (MCFT)"]
        weight = priorities.get(sector, 1.0)

        if demand > 0:
            weighted_satisfaction += weight * allocation / demand
            total_weight += weight

    return {
        "Total Allocated (MCFT)": result["Total Allocated (MCFT)"],
        "Unmet Demand (MCFT)": result["Unmet Demand (MCFT)"],
        "Overall Demand Satisfaction (%)": (
            100 * result["Total Allocated (MCFT)"] / total_demand
            if total_demand > 0 else 100.0
        ),
        "Weighted Satisfaction Score (%)": (
            100 * weighted_satisfaction / total_weight
            if total_weight > 0 else 100.0
        ),
    }


metrics_comparison = pd.DataFrame({
    "Baseline": calculate_metrics(baseline),
    "Optimization": calculate_metrics(optimized),
})

display(metrics_comparison.round(2))


,Baseline,Optimization
Total Allocated (MCFT),80.00,80.00
Unmet Demand (MCFT),30.00,30.00
Overall Demand Satisfaction (%),72.73,72.73
Weighted Satisfaction Score (%),77.78,83.33


In [34]:
import pandas as pd

df = pd.read_csv("data/reservoir_daily.csv")

mettur = df[df["reservoir_slug"] == "mettur"].copy()
mettur["date"] = pd.to_datetime(mettur["date"])
mettur = mettur.sort_values("date").reset_index(drop=True)

mettur["storage_percentage"] = (
    mettur["storage_mcft"] / mettur["capacity_mcft"] * 100
)

print("Mettur records:", len(mettur))
print("Start date:", mettur["date"].min().date())
print("End date:", mettur["date"].max().date())

display(
    mettur[
        ["date", "storage_mcft", "capacity_mcft",
         "inflow_cusecs", "outflow_cusecs", "storage_percentage"]
    ].tail(5)
)

Mettur records: 570
Start date: 2025-01-01
End date: 2026-10-25


,date,storage_mcft,capacity_mcft,inflow_cusecs,outflow_cusecs,storage_percentage
565,2026-10-21,21044.0,93470,4834.0,6003.0,22.514176
566,2026-10-22,20955.0,93470,2702.0,3628.0,22.418958
567,2026-10-23,20887.0,93470,2319.0,3002.0,22.346207
568,2026-10-24,20805.0,93470,1409.0,2254.0,22.258479
569,2026-10-25,20846.0,93470,1582.0,1003.0,22.302343


In [35]:

# Phase 4, Step 1: Prepare daily Mettur simulation inputs

mettur_sim = mettur.copy()

# Define the reserve as a configurable share of reservoir capacity.
# This is a planning assumption, not an official operating rule.
reserve_fraction = 0.20

mettur_sim["reserve_mcft"] = (
    mettur_sim["capacity_mcft"] * reserve_fraction
)

mettur_sim["usable_water_mcft"] = (
    mettur_sim["storage_mcft"] - mettur_sim["reserve_mcft"]
).clip(lower=0)

print("Simulation rows:", len(mettur_sim))
print("Reserve assumption:", f"{reserve_fraction:.0%} of capacity")
print(
    "Dates with storage below assumed reserve:",
    (mettur_sim["storage_mcft"] < mettur_sim["reserve_mcft"]).sum()
)

display(
    mettur_sim[
        [
            "date",
            "storage_mcft",
            "capacity_mcft",
            "reserve_mcft",
            "usable_water_mcft",
        ]
    ].tail(5).round(2)
)


Simulation rows: 570
Reserve assumption: 20% of capacity
Dates with storage below assumed reserve: 0


C:\Users\sahan\AppData\Local\Temp\ipykernel_37928\3221349532.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(5).round(2)


,date,storage_mcft,capacity_mcft,reserve_mcft,usable_water_mcft
565,2026-10-21,21044.0,93470,18694.0,2350.0
566,2026-10-22,20955.0,93470,18694.0,2261.0
567,2026-10-23,20887.0,93470,18694.0,2193.0
568,2026-10-24,20805.0,93470,18694.0,2111.0
569,2026-10-25,20846.0,93470,18694.0,2152.0


In [36]:

from src.allocation import allocate_water, optimize_water_allocation

# Illustrative daily demands (MCFT/day), not measured sector demand
daily_demands = {
    "domestic_demand": 30,
    "agriculture_demand": 60,
    "industry_demand": 20,
}

baseline_rows = []
optimized_rows = []

for _, row in mettur_sim.iterrows():
    inputs = {
        "available_water": row["storage_mcft"],
        "reserve_water": row["reserve_mcft"],
        **daily_demands,
    }

    baseline_result = allocate_water(**inputs)
    optimized_result = optimize_water_allocation(**inputs)

    baseline_rows.append({
        "date": row["date"],
        "storage_mcft": row["storage_mcft"],
        "usable_water_mcft": row["usable_water_mcft"],
        "allocated_mcft": baseline_result["Total Allocated (MCFT)"],
        "unmet_mcft": baseline_result["Unmet Demand (MCFT)"],
        "domestic_allocated": baseline_result["Domestic Allocation (MCFT)"],
        "agriculture_allocated": baseline_result["Agriculture Allocation (MCFT)"],
        "industry_allocated": baseline_result["Industry Allocation (MCFT)"],
    })

    optimized_rows.append({
        "date": row["date"],
        "storage_mcft": row["storage_mcft"],
        "usable_water_mcft": row["usable_water_mcft"],
        "allocated_mcft": optimized_result["Total Allocated (MCFT)"],
        "unmet_mcft": optimized_result["Unmet Demand (MCFT)"],
        "domestic_allocated": optimized_result["Domestic Allocation (MCFT)"],
        "agriculture_allocated": optimized_result["Agriculture Allocation (MCFT)"],
        "industry_allocated": optimized_result["Industry Allocation (MCFT)"],
    })

baseline_daily = pd.DataFrame(baseline_rows)
optimized_daily = pd.DataFrame(optimized_rows)

print("Baseline simulation rows:", len(baseline_daily))
print("Optimization simulation rows:", len(optimized_daily))

display(baseline_daily.tail().round(2))
display(optimized_daily.tail().round(2))


Baseline simulation rows: 570
Optimization simulation rows: 570


C:\Users\sahan\AppData\Local\Temp\ipykernel_37928\4083615948.py:51: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(baseline_daily.tail().round(2))


,date,storage_mcft,usable_water_mcft,allocated_mcft,unmet_mcft,domestic_allocated,agriculture_allocated,industry_allocated
565,2026-10-21,21044.0,2350.0,110.0,0.0,30.0,60.0,20.0
566,2026-10-22,20955.0,2261.0,110.0,0.0,30.0,60.0,20.0
567,2026-10-23,20887.0,2193.0,110.0,0.0,30.0,60.0,20.0
568,2026-10-24,20805.0,2111.0,110.0,0.0,30.0,60.0,20.0
569,2026-10-25,20846.0,2152.0,110.0,0.0,30.0,60.0,20.0


C:\Users\sahan\AppData\Local\Temp\ipykernel_37928\4083615948.py:52: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(optimized_daily.tail().round(2))


,date,storage_mcft,usable_water_mcft,allocated_mcft,unmet_mcft,domestic_allocated,agriculture_allocated,industry_allocated
565,2026-10-21,21044.0,2350.0,110.0,0.0,30.0,60.0,20.0
566,2026-10-22,20955.0,2261.0,110.0,0.0,30.0,60.0,20.0
567,2026-10-23,20887.0,2193.0,110.0,0.0,30.0,60.0,20.0
568,2026-10-24,20805.0,2111.0,110.0,0.0,30.0,60.0,20.0
569,2026-10-25,20846.0,2152.0,110.0,0.0,30.0,60.0,20.0


In [38]:
from src.allocation import allocate_water, optimize_water_allocation

# Illustrative sector demands in MCFT/day — assumptions, not measured data
daily_demands = {
    "domestic_demand": 30,
    "agriculture_demand": 60,
    "industry_demand": 20,
}

baseline_rows = []
optimized_rows = []

for _, row in mettur_sim.iterrows():

    # Daily allocation budget cannot exceed observed outflow
    available = row["observed_outflow_mcft_day"]

    inputs = {
        "available_water": available,
        "reserve_water": 0,
        **daily_demands,
    }

    baseline = allocate_water(**inputs)
    optimized = optimize_water_allocation(**inputs)

    baseline_rows.append({
        "date": row["date"],
        "available_mcft_day": available,
        "allocated_mcft_day": baseline["Total Allocated (MCFT)"],
        "unmet_mcft_day": baseline["Unmet Demand (MCFT)"],
    })

    optimized_rows.append({
        "date": row["date"],
        "available_mcft_day": available,
        "allocated_mcft_day": optimized["Total Allocated (MCFT)"],
        "unmet_mcft_day": optimized["Unmet Demand (MCFT)"],
    })

baseline_daily = pd.DataFrame(baseline_rows)
optimized_daily = pd.DataFrame(optimized_rows)

print("Baseline records:", len(baseline_daily))
print("Optimized records:", len(optimized_daily))

display(baseline_daily.tail(5).round(2))
display(optimized_daily.tail(5).round(2))

Baseline records: 570
Optimized records: 570


C:\Users\sahan\AppData\Local\Temp\ipykernel_37928\4227270440.py:47: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(baseline_daily.tail(5).round(2))


,date,available_mcft_day,allocated_mcft_day,unmet_mcft_day
565,2026-10-21,518.66,110.00,0.00
566,2026-10-22,313.46,110.00,0.00
567,2026-10-23,259.37,110.00,0.00
568,2026-10-24,194.75,110.00,0.00
569,2026-10-25,86.66,86.66,23.34


C:\Users\sahan\AppData\Local\Temp\ipykernel_37928\4227270440.py:48: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(optimized_daily.tail(5).round(2))


,date,available_mcft_day,allocated_mcft_day,unmet_mcft_day
565,2026-10-21,518.66,110.00,0.00
566,2026-10-22,313.46,110.00,0.00
567,2026-10-23,259.37,110.00,0.00
568,2026-10-24,194.75,110.00,0.00
569,2026-10-25,86.66,86.66,23.34


In [39]:
# Phase 4: Compare model performance across all observations

comparison = pd.DataFrame({
    "Metric": [
        "Average daily water available (MCFT/day)",
        "Average daily water allocated (MCFT/day)",
        "Average daily unmet demand (MCFT/day)",
        "Days with unmet demand",
    ],
    "Baseline": [
        baseline_daily["available_mcft_day"].mean(),
        baseline_daily["allocated_mcft_day"].mean(),
        baseline_daily["unmet_mcft_day"].mean(),
        (baseline_daily["unmet_mcft_day"] > 0).sum(),
    ],
    "Optimized": [
        optimized_daily["available_mcft_day"].mean(),
        optimized_daily["allocated_mcft_day"].mean(),
        optimized_daily["unmet_mcft_day"].mean(),
        (optimized_daily["unmet_mcft_day"] > 0).sum(),
    ],
})

display(comparison.round(2))

,Metric,Baseline,Optimized
0,Average daily water available (MCFT/day),618.27,618.27
1,Average daily water allocated (MCFT/day),99.09,99.09
2,Average daily unmet demand (MCFT/day),10.91,10.91
3,Days with unmet demand,216.00,216.00


In [40]:
from src.uncertainty import simulate_rainfall_scenarios

help(simulate_rainfall_scenarios)

Help on function simulate_rainfall_scenarios in module src.uncertainty:

simulate_rainfall_scenarios(current_storage, reserve_water, expected_inflow)
    Simulate water availability under three illustrative
    rainfall scenarios. All water quantities are in TMC.



In [45]:
from src.uncertainty import simulate_rainfall_scenarios

scenarios = simulate_rainfall_scenarios(
    current_storage_mcft=20846,
    reserve_water_mcft=18694,
    expected_inflow_mcft=100
)

display(pd.DataFrame(scenarios))

,Scenario,Scenario Inflow (MCFT),Available Water (MCFT),Usable Water (MCFT)
0,Drought,40.0,20886.0,2192.0
1,Normal,100.0,20946.0,2252.0
2,Wet,160.0,21006.0,2312.0


In [44]:
import importlib
import src.uncertainty as uncertainty

importlib.reload(uncertainty)

help(uncertainty.simulate_rainfall_scenarios)

Help on function simulate_rainfall_scenarios in module src.uncertainty:

simulate_rainfall_scenarios(current_storage_mcft, reserve_water_mcft, expected_inflow_mcft)
    Compare illustrative rainfall scenarios for one period.
    All water quantities are in MCFT.

    Scenario multipliers are assumptions, not forecasts.



In [48]:
from src.allocation import optimize_fair_allocation

fair_result = optimize_fair_allocation(
    available_water=86.66,
    domestic_demand=30,
    agriculture_demand=60,
    industry_demand=20,
    reserve_water=0
)

display(pd.DataFrame([fair_result]).round(2))

,Domestic,Agriculture,Industry,Minimum Satisfaction (%)
0,23.63,47.27,15.76,78.78


In [47]:
import importlib
import src.allocation as allocation

importlib.reload(allocation)

print(hasattr(allocation, "optimize_fair_allocation"))

True


In [49]:

from src.allocation import allocate_water, optimize_fair_allocation
import numpy as np
import pandas as pd

demands = {
    "domestic_demand": 30,
    "agriculture_demand": 60,
    "industry_demand": 20,
}

sectors = ["Domestic", "Agriculture", "Industry"]
demand_values = np.array([30.0, 60.0, 20.0])

results = []

for _, row in mettur_sim.iterrows():
    available = row["observed_outflow_mcft_day"]

    # Only evaluate days when total demand cannot be met
    if available >= demand_values.sum():
        continue

    baseline = allocate_water(
        available_water=available,
        reserve_water=0,
        **demands
    )

    fair = optimize_fair_allocation(
        available_water=available,
        reserve_water=0,
        **demands
    )

    baseline_allocations = np.array([
        baseline["Domestic Allocation (MCFT)"],
        baseline["Agriculture Allocation (MCFT)"],
        baseline["Industry Allocation (MCFT)"],
    ])

    fair_allocations = np.array([
        fair["Domestic"],
        fair["Agriculture"],
        fair["Industry"],
    ])

    results.append({
        "date": row["date"],
        "baseline_domestic_pct": 100 * baseline_allocations[0] / 30,
        "baseline_agriculture_pct": 100 * baseline_allocations[1] / 60,
        "baseline_industry_pct": 100 * baseline_allocations[2] / 20,
        "baseline_min_satisfaction_pct": min(
            100 * baseline_allocations / demand_values
        ),
        "fair_domestic_pct": 100 * fair_allocations[0] / 30,
        "fair_agriculture_pct": 100 * fair_allocations[1] / 60,
        "fair_industry_pct": 100 * fair_allocations[2] / 20,
        "fair_min_satisfaction_pct": fair["Minimum Satisfaction (%)"],
    })

shortage_comparison = pd.DataFrame(results)

metrics = pd.DataFrame({
    "Metric": [
        "Average domestic demand satisfied (%)",
        "Average agriculture demand satisfied (%)",
        "Average industry demand satisfied (%)",
        "Average minimum sector satisfaction (%)",
    ],
    "Baseline": [
        shortage_comparison["baseline_domestic_pct"].mean(),
        shortage_comparison["baseline_agriculture_pct"].mean(),
        shortage_comparison["baseline_industry_pct"].mean(),
        shortage_comparison["baseline_min_satisfaction_pct"].mean(),
    ],
    "Maximin Fair": [
        shortage_comparison["fair_domestic_pct"].mean(),
        shortage_comparison["fair_agriculture_pct"].mean(),
        shortage_comparison["fair_industry_pct"].mean(),
        shortage_comparison["fair_min_satisfaction_pct"].mean(),
    ],
})

print("Shortage days evaluated:", len(shortage_comparison))
display(metrics.round(2))


Shortage days evaluated: 216


,Metric,Baseline,Maximin Fair
0,Average domestic demand satisfied (%),100.00,73.84
1,Average agriculture demand satisfied (%),84.11,73.84
2,Average industry demand satisfied (%),3.79,73.84
3,Average minimum sector satisfaction (%),3.79,73.84


In [50]:
import importlib
import src.uncertainty as uncertainty

importlib.reload(uncertainty)

scenarios = uncertainty.simulate_rainfall_scenarios(
    current_storage_mcft=20846,
    capacity_mcft=93470,
    reserve_water_mcft=18694,
    expected_inflow_mcft=100
)

display(pd.DataFrame(scenarios))

,Scenario,Scenario Inflow (MCFT),Projected Storage (MCFT),Usable Storage (MCFT)
0,Drought,40.0,20886.0,2192.0
1,Normal,100.0,20946.0,2252.0
2,Wet,160.0,21006.0,2312.0


In [51]:
from src.allocation import optimize_fair_allocation

scenario_results = []

for scenario in scenarios:
    # Illustrative demands for one planning period (MCFT)
    result = optimize_fair_allocation(
        available_water=scenario["Usable Storage (MCFT)"],
        domestic_demand=30,
        agriculture_demand=60,
        industry_demand=20,
        reserve_water=0
    )

    scenario_results.append({
        "Scenario": scenario["Scenario"],
        "Usable Water (MCFT)": scenario["Usable Storage (MCFT)"],
        "Domestic Allocation (MCFT)": result["Domestic"],
        "Agriculture Allocation (MCFT)": result["Agriculture"],
        "Industry Allocation (MCFT)": result["Industry"],
        "Minimum Satisfaction (%)": result["Minimum Satisfaction (%)"]
    })

scenario_comparison = pd.DataFrame(scenario_results)
display(scenario_comparison.round(2))

,Scenario,Usable Water (MCFT),Domestic Allocation (MCFT),Agriculture Allocation (MCFT),Industry Allocation (MCFT),Minimum Satisfaction (%)
0,Drought,2192.0,30.0,60.0,20.0,100.0
1,Normal,2252.0,30.0,60.0,20.0,100.0
2,Wet,2312.0,30.0,60.0,20.0,100.0


In [52]:
stress_results = []

for scenario in scenarios:
    result = optimize_fair_allocation(
        available_water=scenario["Usable Storage (MCFT)"],
        domestic_demand=1000,
        agriculture_demand=2000,
        industry_demand=1000,
        reserve_water=0
    )

    stress_results.append({
        "Scenario": scenario["Scenario"],
        "Usable Water (MCFT)": scenario["Usable Storage (MCFT)"],
        "Domestic Allocation": result["Domestic"],
        "Agriculture Allocation": result["Agriculture"],
        "Industry Allocation": result["Industry"],
        "Minimum Satisfaction (%)": result["Minimum Satisfaction (%)"]
    })

display(pd.DataFrame(stress_results).round(2))

,Scenario,Usable Water (MCFT),Domestic Allocation,Agriculture Allocation,Industry Allocation,Minimum Satisfaction (%)
0,Drought,2192.0,548.0,1096.0,548.0,54.8
1,Normal,2252.0,563.0,1126.0,563.0,56.3
2,Wet,2312.0,578.0,1156.0,578.0,57.8


In [53]:

import numpy as np
import pandas as pd

from src.allocation import allocate_water, optimize_fair_allocation

# Existing project assumptions — MCFT per day
demands = {
    "domestic_demand": 30,
    "agriculture_demand": 60,
    "industry_demand": 20
}

sectors = {
    "Domestic": 30,
    "Agriculture": 60,
    "Industry": 20
}

results = []

for _, row in mettur_sim.iterrows():
    available = row["observed_outflow_mcft_day"]

    # Method 1: Existing priority-based baseline
    baseline = allocate_water(
        available_water=available,
        reserve_water=0,
        **demands
    )

    # Method 2: Maximin fair allocation
    fair = optimize_fair_allocation(
        available_water=available,
        reserve_water=0,
        **demands
    )

    record = {"date": row["date"]}

    for sector, demand in sectors.items():
        baseline_alloc = baseline[f"{sector} Allocation (MCFT)"]
        fair_alloc = fair[sector]

        record[f"Baseline {sector} (%)"] = (
            100 * baseline_alloc / demand if demand > 0 else 100
        )
        record[f"Fair {sector} (%)"] = (
            100 * fair_alloc / demand if demand > 0 else 100
        )

    record["Baseline Minimum Satisfaction (%)"] = min(
        record[f"Baseline {s} (%)"] for s in sectors
    )
    record["Fair Minimum Satisfaction (%)"] = fair[
        "Minimum Satisfaction (%)"
    ]

    record["Baseline Industry Zero"] = (
        baseline["Industry Allocation (MCFT)"] <= 1e-8
    )
    record["Fair Industry Zero"] = fair["Industry"] <= 1e-8

    results.append(record)

historical_results = pd.DataFrame(results)

summary = pd.DataFrame({
    "Metric": [
        "Average domestic satisfaction (%)",
        "Average agriculture satisfaction (%)",
        "Average industry satisfaction (%)",
        "Average minimum sector satisfaction (%)",
        "Days industry received zero water"
    ],
    "Priority Baseline": [
        historical_results["Baseline Domestic (%)"].mean(),
        historical_results["Baseline Agriculture (%)"].mean(),
        historical_results["Baseline Industry (%)"].mean(),
        historical_results["Baseline Minimum Satisfaction (%)"].mean(),
        historical_results["Baseline Industry Zero"].sum()
    ],
    "Maximin Fair": [
        historical_results["Fair Domestic (%)"].mean(),
        historical_results["Fair Agriculture (%)"].mean(),
        historical_results["Fair Industry (%)"].mean(),
        historical_results["Fair Minimum Satisfaction (%)"].mean(),
        historical_results["Fair Industry Zero"].sum()
    ]
})

print("Recorded observations evaluated:", len(historical_results))
display(summary.round(2))


Recorded observations evaluated: 570


,Metric,Priority Baseline,Maximin Fair
0,Average domestic satisfaction (%),100.00,90.09
1,Average agriculture satisfaction (%),93.98,90.09
2,Average industry satisfaction (%),63.54,90.09
3,Average minimum sector satisfaction (%),63.54,90.09
4,Days industry received zero water,202.00,0.00


In [56]:

# Improved dynamic simulation:
# rolling 30-calendar-day inflow scenarios, reserve sensitivity,
# and sector-level demand satisfaction.

import numpy as np
import pandas as pd

from src.allocation import allocate_water, optimize_fair_allocation

CAPACITY = float(mettur_sim["capacity_mcft"].max())
START_STORAGE = float(
    mettur_sim.sort_values("date").iloc[-1]["storage_mcft"]
)
PLANNING_DAYS = 30

SECTOR_DEMANDS = {
    "domestic_demand": 30.0,
    "agriculture_demand": 60.0,
    "industry_demand": 20.0
}

SECTOR_NAMES = {
    "Domestic": "Domestic",
    "Agriculture": "Agriculture",
    "Industry": "Industry"
}

# Build a calendar-day series. Missing observations remain missing;
# they are NOT treated as zero inflow.
inflow_daily = (
    mettur_sim.assign(
        date=pd.to_datetime(mettur_sim["date"]),
        inflow_mcft_day=(
            pd.to_numeric(mettur_sim["inflow_cusecs"], errors="coerce")
            * 86400 / 1_000_000
        )
    )
    .set_index("date")["inflow_mcft_day"]
    .sort_index()
    .resample("D")
    .mean()
)

# Keep only complete 30-calendar-day windows.
rolling_30d_inflow = inflow_daily.rolling(
    window=30, min_periods=30
).sum().dropna()

if rolling_30d_inflow.empty:
    raise ValueError(
        "No complete 30-day inflow windows. Check missing dates in the dataset."
    )

# Derive scenarios from historical 30-day totals.
scenario_totals = {
    "Drought (10th percentile)": rolling_30d_inflow.quantile(0.10),
    "Normal (50th percentile)": rolling_30d_inflow.quantile(0.50),
    "Wet (90th percentile)": rolling_30d_inflow.quantile(0.90)
}

# Convert each 30-day total to a constant daily average for the stress test.
scenario_daily_inflow = {
    name: total / PLANNING_DAYS
    for name, total in scenario_totals.items()
}

simulation_rows = []

for reserve_fraction in [0.10, 0.20]:
    reserve = reserve_fraction * CAPACITY

    for scenario, daily_inflow in scenario_daily_inflow.items():
        for method in ["Priority Baseline", "Maximin Fair"]:

            storage = START_STORAGE

            for day in range(1, PLANNING_DAYS + 1):

                # Add inflow, cap storage, and account for overflow first.
                storage_before_release = storage + daily_inflow
                overflow = max(0.0, storage_before_release - CAPACITY)
                available_for_release = min(
                    CAPACITY, storage_before_release
                )

                if method == "Priority Baseline":
                    allocation = allocate_water(
                        available_water=available_for_release,
                        reserve_water=reserve,
                        **SECTOR_DEMANDS
                    )

                    domestic = allocation["Domestic Allocation (MCFT)"]
                    agriculture = allocation["Agriculture Allocation (MCFT)"]
                    industry = allocation["Industry Allocation (MCFT)"]

                else:
                    allocation = optimize_fair_allocation(
                        available_water=available_for_release,
                        reserve_water=reserve,
                        **SECTOR_DEMANDS
                    )

                    domestic = allocation["Domestic"]
                    agriculture = allocation["Agriculture"]
                    industry = allocation["Industry"]

                release = domestic + agriculture + industry
                ending_storage = available_for_release - release

                # Satisfaction is based on the assumed daily sector demand.
                simulation_rows.append({
                    "Reserve (%)": int(reserve_fraction * 100),
                    "Scenario": scenario,
                    "Method": method,
                    "Day": day,
                    "Daily Inflow (MCFT)": daily_inflow,
                    "Starting Storage (MCFT)": storage,
                    "Domestic Release (MCFT)": domestic,
                    "Agriculture Release (MCFT)": agriculture,
                    "Industry Release (MCFT)": industry,
                    "Domestic Satisfaction (%)": 100 * domestic / 30,
                    "Agriculture Satisfaction (%)": 100 * agriculture / 60,
                    "Industry Satisfaction (%)": 100 * industry / 20,
                    "Total Release (MCFT)": release,
                    "Ending Storage (MCFT)": ending_storage,
                    "Overflow (MCFT)": overflow
                })

                storage = ending_storage

dynamic_results = pd.DataFrame(simulation_rows)

print("Historical rolling 30-day inflow totals (MCFT):")
display(pd.DataFrame([
    {
        "Scenario": name,
        "Historical 30-day Inflow (MCFT)": total,
        "Assumed Daily Inflow (MCFT/day)": total / PLANNING_DAYS
    }
    for name, total in scenario_totals.items()
]).round(2))

print("Sector satisfaction and storage summary:")
display(
    dynamic_results.groupby(
        ["Reserve (%)", "Scenario", "Method"]
    ).agg(
        Domestic_Satisfaction=("Domestic Satisfaction (%)", "mean"),
        Agriculture_Satisfaction=("Agriculture Satisfaction (%)", "mean"),
        Industry_Satisfaction=("Industry Satisfaction (%)", "mean"),
        Ending_Storage=("Ending Storage (MCFT)", "last"),
        Total_Overflow=("Overflow (MCFT)", "sum")
    ).round(2)
)

print("Simulation rows:", len(dynamic_results))
print(
    "Reserve violations:",
    (dynamic_results["Ending Storage (MCFT)"]
     < dynamic_results["Reserve (%)"].map(
         lambda p: p / 100 * CAPACITY
     ) - 1e-6).sum()
)
print(
    "Negative storage records:",
    (dynamic_results["Ending Storage (MCFT)"] < -1e-6).sum()
)


Historical rolling 30-day inflow totals (MCFT):


,Scenario,Historical 30-day Inflow (MCFT),Assumed Daily Inflow (MCFT/day)
0,Drought (10th percentile),753.72,25.12
1,Normal (50th percentile),1988.06,66.27
2,Wet (90th percentile),61954.12,2065.14


Sector satisfaction and storage summary:


Domestic_Satisfaction  \
Reserve (%) Scenario                  Method                                     
10          Drought (10th percentile) Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
            Normal (50th percentile)  Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
            Wet (90th percentile)     Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
20          Drought (10th percentile) Maximin Fair                       88.05   
                                      Priority Baseline                  97.83   
            Normal (50th percentile)  Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
            Wet (90th percentile)     Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   

                                                         Agriculture_Satisfaction  \
Reserve (%) Scenario                  Method                                        
10          Drought (10th percentile) Maximin Fair                         100.00   
                                      Priority Baseline                    100.00   
            Normal (50th percentile)  Maximin Fair                         100.00   
                                      Priority Baseline                    100.00   
            Wet (90th percentile)     Maximin Fair                         100.00   
                                      Priority Baseline                    100.00   
20          Drought (10th percentile) Maximin Fair                          88.05   
                                      Priority Baseline                     84.73   
            Normal (50th percentile)  Maximin Fair                         100.00   
                                      Priority Baseline                    100.00   
            Wet (90th percentile)     Maximin Fair                         100.00   
                                      Priority Baseline                    100.00   

                                                         Industry_Satisfaction  \
Reserve (%) Scenario                  Method                                     
10          Drought (10th percentile) Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
            Normal (50th percentile)  Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
            Wet (90th percentile)     Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
20          Drought (10th percentile) Maximin Fair                       88.05   
                                      Priority Baseline                  83.33   
            Normal (50th percentile)  Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   
            Wet (90th percentile)     Maximin Fair                      100.00   
                                      Priority Baseline                 100.00   

                                                         Ending_Storage  \
Reserve (%) Scenario                  Method                              
10          Drought (10th percentile) Maximin Fair             18299.72   
                                      Priority Baseline        18299.72   
            Normal (50th percentile)  Maximin Fair             19534.06   
                                      Priority Baseline        19534.06   
            Wet (90th percentile)     Maximin Fair             79500.12   
                                      Prior

Simulation rows: 360
Reserve violations: 0
Negative storage records: 0


In [55]:

# Simulation validation and sector comparison

reserve_violations = (
    dynamic_results["Ending Storage (MCFT)"] < RESERVE - 1e-6
).sum()

negative_storage = (
    dynamic_results["Ending Storage (MCFT)"] < -1e-6
).sum()

# Compare allocation policies for each scenario
sector_summary = dynamic_results.groupby(
    ["Scenario", "Method"]
).agg(
    Domestic_Total=("Domestic Release (MCFT)", "sum"),
    Agriculture_Total=("Agriculture Release (MCFT)", "sum"),
    Industry_Total=("Industry Release (MCFT)", "sum"),
    Total_Release=("Total Release (MCFT)", "sum"),
    Final_Storage=("Ending Storage (MCFT)", "last")
).round(2)

print("Days simulated:", len(dynamic_results))
print("Reserve violations:", reserve_violations)
print("Negative-storage records:", negative_storage)
print("\nTotal sector releases over 30 days (MCFT):")
display(sector_summary)


Days simulated: 180
Reserve violations: 0
Negative-storage records: 0

Total sector releases over 30 days (MCFT):


Domestic_Total  \
Scenario                  Method                              
Drought (10th percentile) Maximin Fair               641.06   
                          Priority Baseline          689.56   
Normal (50th percentile)  Maximin Fair               900.00   
                          Priority Baseline          900.00   
Wet (90th percentile)     Maximin Fair               900.00   
                          Priority Baseline          900.00   

                                             Agriculture_Total  \
Scenario                  Method                                 
Drought (10th percentile) Maximin Fair                 1282.12   
                          Priority Baseline            1260.00   
Normal (50th percentile)  Maximin Fair                 1800.00   
                          Priority Baseline            1800.00   
Wet (90th percentile)     Maximin Fair                 1800.00   
                          Priority Baseline            1800.00   

                                             Industry_Total  Total_Release  \
Scenario                  Method                                             
Drought (10th percentile) Maximin Fair               427.37        2350.55   
                          Priority Baseline          400.98        2350.55   
Normal (50th percentile)  Maximin Fair               600.00        3300.00   
                          Priority Baseline          600.00        3300.00   
Wet (90th percentile)     Maximin Fair               600.00        3300.00   
                          Priority Baseline          600.00        3300.00   

                                             Final_Storage  
Scenario                  Method                            
Drought (10th percentile) Maximin Fair            18694.00  
                          Priority Baseline       18694.00  
Normal (50th percentile)  Maximin Fair            20870.24  
                          Priority Baseline       20870.24  
Wet (90th percentile)     Maximin Fair            62969.50  
                          Priority Baseline       62969.50

In [57]:

# Validate the data-derived 30-day inflow scenarios

rolling_summary = rolling_30d_inflow.describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("Historical rolling 30-day inflow distribution (MCFT):")
display(rolling_summary.round(2).to_frame("Inflow (MCFT)"))

print("Largest historical rolling 30-day inflow:")
print(round(rolling_30d_inflow.max(), 2), "MCFT")

print("\nScenario percentile checks:")
for name, value in scenario_totals.items():
    percentile = (
        rolling_30d_inflow.le(value).mean() * 100
    )
    print(f"{name}: {value:,.2f} MCFT — percentile rank {percentile:.1f}%")

# Compare scenario inflows with reservoir capacity
print("\nReservoir capacity:", round(CAPACITY, 2), "MCFT")
print(
    "Wet scenario inflow as % of capacity:",
    round(100 * scenario_totals["Wet (90th percentile)"] / CAPACITY, 2),
    "%"
)


Historical rolling 30-day inflow distribution (MCFT):


,Inflow (MCFT)
count,187.00
mean,14418.94
std,24522.48
min,190.43
10%,753.72
25%,1019.95
50%,1988.06
75%,16852.71
90%,61954.12
95%,79333.73


Largest historical rolling 30-day inflow:
86980.26 MCFT

Scenario percentile checks:
Drought (10th percentile): 753.72 MCFT — percentile rank 10.2%
Normal (50th percentile): 1,988.06 MCFT — percentile rank 50.3%
Wet (90th percentile): 61,954.12 MCFT — percentile rank 89.8%

Reservoir capacity: 93470.0 MCFT
Wet scenario inflow as % of capacity: 66.28 %
